# Qualifying studies and credible sets

Quality control that defines the analysis set. Methods "Qualified studies and CSs".

Disease studies: cases fewer than controls, not a measurement, at least 1,000 samples,
case fraction at least 0.001, one problematic publication removed.
Measurement studies: measurement traits, cases not fewer than controls, protein and
microbiome traits removed.
Credible sets: expected minor allele count at least 20, |rescaled beta| at most 3, and
rare sets (MAF <= 0.01) only if replicated, colocalising with a molQTL, or carrying a PAV.

Writes `qualifying_gwas_studies`, `qualifying_measurement_studies`,
`qualifying_credible_sets`, `qualifying_measurement_credible_sets`.

In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 00:26:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
studies = session.spark.read.parquet(paper.derived("study_therapeutic_areas")).cache()
lead_variant_effect = session.spark.read.parquet(paper.derived("lead_variant_effect"))
disease = session.spark.read.parquet(paper.release("disease") + "/disease.parquet")

26/08/19 00:26:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Qualifying studies

In [3]:
qualifying_studies = (
    studies.filter(f.col("binaryLessCases"))
    .filter(~f.col("measurement"))
    .filter(f.col("nSamples") >= 1000)
    .filter((f.col("nCases") / f.col("nSamples")) >= 0.001)
    .filter((~f.col("pubmedId").isin(["40069456"])) | f.col("pubmedId").isNull())
)
qualifying_studies.write.mode("overwrite").parquet(paper.derived("qualifying_gwas_studies"))
print("qualifying disease studies:", session.spark.read.parquet(paper.derived("qualifying_gwas_studies")).count())

qualifying disease studies: 15730


In [4]:
# EFO_0007882 microbiome, EFO_0004747 protein measurement, and their descendants
excluded = disease.filter(f.col("id").isin(["EFO_0007882", "EFO_0004747"])).select(f.explode("descendants").alias("id"))
excluded_ids = [row["id"] for row in excluded.collect()] + ["EFO_0007882", "EFO_0004747"]

qualifying_measurements = (
    studies.filter(f.col("measurement"))
    .filter(~f.col("binaryLessCases"))
    .filter(f.size(f.array_intersect("diseaseIds", f.lit(excluded_ids))) == 0)
)
qualifying_measurements.write.mode("overwrite").parquet(paper.derived("qualifying_measurement_studies"))
print(
    "qualifying measurement studies:",
    session.spark.read.parquet(paper.derived("qualifying_measurement_studies")).count(),
)

qualifying measurement studies: 61885


## Functional-genomics support per credible set, used to rescue rare sets

In [5]:
replicated = session.spark.read.parquet(paper.derived("replicated_gwas_cs")).withColumn("replicated", f.lit(1))

features = (
    session.spark.read.parquet(paper.release("l2g_feature_matrix"))
    .groupBy("studyLocusId")
    .agg(
        f.max("eQtlColocH4Maximum").alias("eqtlH4"),
        f.max("eQtlColocClppMaximum").alias("eqtlCLPP"),
        f.max("pQtlColocH4Maximum").alias("pqtlH4"),
        f.max("pQtlColocClppMaximum").alias("pqtlCLPP"),
        f.max("sQtlColocH4Maximum").alias("sqtlH4"),
        f.max("sQtlColocClppMaximum").alias("sqtlCLPP"),
        f.max("vepMaximum").alias("vep"),
    )
    .join(replicated, "studyLocusId", "outer")
    .fillna({"replicated": 0})
    .persist()
)

supported = features.filter(
    (f.col("eqtlH4") >= 0.8)
    | (f.col("eqtlCLPP") >= 0.01)
    | (f.col("pqtlH4") >= 0.8)
    | (f.col("pqtlCLPP") >= 0.01)
    | (f.col("sqtlH4") >= 0.8)
    | (f.col("sqtlCLPP") >= 0.01)
    | (f.col("vep") >= 0.66)
    | (f.col("replicated") == 1)
)

## Qualifying credible sets

In [6]:
def qualify(study_table, count_column):
    """Credible sets of the given studies passing power, effect-size and rare-variant QC."""
    first = (
        lead_variant_effect.join(study_table.select("studyId", count_column), "studyId", "inner")
        .filter(2 * f.col("majorLdPopulationMaf.value") * f.col(count_column) >= 20)
        .filter(f.abs("rescaledStatistics.absEstimatedBeta") <= 3)
        .persist()
    )
    common = first.filter(f.col("majorLdPopulationMaf.value") > 0.01)
    rare = first.filter(f.col("majorLdPopulationMaf.value") <= 0.01).join(supported, "studyLocusId", "semi")
    return common.unionByName(rare)


qualify(session.spark.read.parquet(paper.derived("qualifying_gwas_studies")), "nCases").write.mode("overwrite").parquet(
    paper.derived("qualifying_credible_sets")
)
qualify(session.spark.read.parquet(paper.derived("qualifying_measurement_studies")), "nSamples").write.mode(
    "overwrite"
).parquet(paper.derived("qualifying_measurement_credible_sets"))

disease_cs = session.spark.read.parquet(paper.derived("qualifying_credible_sets")).count()
measurement_cs = session.spark.read.parquet(paper.derived("qualifying_measurement_credible_sets")).count()
print("qualifying disease CSs:", disease_cs)
print("qualifying measurement CSs:", measurement_cs)
print("total:", disease_cs + measurement_cs)

qualifying disease CSs: 70618
qualifying measurement CSs: 450357
total: 520975


## Cross-check against the pre-refactor tables

In [7]:
for name in [
    "qualifying_gwas_studies",
    "qualifying_measurement_studies",
    "qualifying_credible_sets",
    "qualifying_measurement_credible_sets",
]:
    print(
        name,
        session.spark.read.parquet(paper.derived(name)).count(),
        session.spark.read.parquet(paper.baseline(name)).count(),
    )

qualifying_gwas_studies 15730 15730


qualifying_measurement_studies 61885 61885


qualifying_credible_sets 70618 70618


qualifying_measurement_credible_sets 450357 450357
